# Phase 2 — PAD-UFES-20 Data Preparation (ResNet50 baseline, Target A)

This notebook does **data prep only**. It does NOT train. It runs:
1. environment setup + version capture
2. Google Drive mount + data-location check
3. dataset validation report
4. frozen 70/15/15 patient split + leakage check
5. a data-loading smoke test (one batch + one forward pass, no training)

**Before running**, download PAD-UFES-20 from Mendeley (`zr7vgbcyr2`) and place it in Drive (see README `Drive layout`).

## 1. Setup + version capture

In [ ]:
# Point this at where you put the project in Drive:
PROJECT_IN_DRIVE = '/content/drive/MyDrive/skin-cancer-phase1'

import sys, subprocess, os
# Light deps only; torch/torchvision are preinstalled on Colab (do NOT reinstall).
subprocess.run([sys.executable,'-m','pip','install','-q',
                'pandas>=2.0','scikit-learn>=1.3','pillow>=9.0','pyyaml>=6.0','tqdm>=4.65'], check=True)
import torch, torchvision, numpy, pandas, sklearn, PIL
snap = (f"torch={torch.__version__}\ntorchvision={torchvision.__version__}\n"
        f"numpy={numpy.__version__}\npandas={pandas.__version__}\n"
        f"sklearn={sklearn.__version__}\npillow={PIL.__version__}\n"
        f"cuda_available={torch.cuda.is_available()}\n"
        f"gpu={torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'}\n")
print(snap)

## 2. Mount Drive + locate data

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
os.makedirs(PROJECT_IN_DRIVE, exist_ok=True)
for sub in ['src','data','splits','results','logs','env']:
    os.makedirs(os.path.join(PROJECT_IN_DRIVE, sub), exist_ok=True)

# Save the env snapshot next to the code for reproducibility
with open(os.path.join(PROJECT_IN_DRIVE,'env','env_snapshot.txt'),'w') as f:
    f.write(snap)

# Sanity-check the expected data layout (doubled nesting!)
import glob
data = os.path.join(PROJECT_IN_DRIVE,'data')
print('metadata.csv present:', os.path.exists(os.path.join(data,'metadata.csv')))
for i in (1,2,3):
    p = os.path.join(data, f'imgs_part_{i}', f'imgs_part_{i}')
    n = len(glob.glob(os.path.join(p,'*'))) if os.path.isdir(p) else 'DIR MISSING'
    print(f'imgs_part_{i}/imgs_part_{i}:', n)

If any `imgs_part_i/imgs_part_i` shows `DIR MISSING`, your extraction nesting is wrong. The three Mendeley zips must each extract so that the images sit at `data/imgs_part_i/imgs_part_i/*.png`.

## 3. Validate the dataset

In [ ]:
# The src/ files must already be copied into PROJECT_IN_DRIVE/src (see README).
import sys
sys.path.insert(0, os.path.join(PROJECT_IN_DRIVE,'src'))
%cd $PROJECT_IN_DRIVE/src
!python validate_dataset.py

## 4. Freeze the 70/15/15 patient split + leakage check

In [ ]:
!python make_split.py

## 5. Data-loading smoke test (NO training)
One batch from each split + one forward pass through ResNet50. Stops before any optimisation.

In [ ]:
!python smoke_test.py

## Done — Phase 2 complete
Check `results/validation_report.txt`, `results/split_report.txt`, `results/smoke_report.txt` and the frozen `splits/padufes20_patient_split.csv`.

**Do not proceed to training without review.** Training is Phase 3.